## Import libraries

In [1]:
import time
import re
import io

import os, sys, contextlib

import pandas as pd
import numpy as np


import speech_recognition as sr
from gtts import gTTS

from pydub import AudioSegment
from scipy.io.wavfile import read, write

import audio2face_pb2
import audio2face_pb2_grpc

import grpc
from audio2face_streaming_utils import push_audio_track

from typing import Union, Type

In [2]:
import google.protobuf
print(google.protobuf.__version__)

3.20.0


In [3]:
@contextlib.contextmanager
def ignoreStderr():
    devnull = os.open(os.devnull, os.O_WRONLY)
    old_stderr = os.dup(2)
    sys.stderr.flush()
    os.dup2(devnull, 2)
    os.close(devnull)
    try:
        yield
    finally:
        os.dup2(old_stderr, 2)
        os.close(old_stderr)

## 1. NLU by ChatGPT
We utilize ChatGPT as the cognitive language understanding engine, leveraging the user-friendly API called revCHATGPT. Users only need to possess a ChatGPT account and provide their access_token to access the functionalities of ChatGPT.With this setup, users can easily interact with ChatGPT by simply using their access_token. You can get your access_token <a href="https://chat.openai.com/api/auth/session"> here.</a>

In [ ]:
# Import the Chatbot class from the revChatGPT package (assuming it exists)
from revChatGPT.V1 import Chatbot

# Set the access_token variable to None initially
# Set OPENAI_API_KEY in your environment before running this cell.
access_token = os.getenv("OPENAI_API_KEY")
# Use an assert statement to check if the access_token variable is not None
assert access_token is not None, "Please make sure you have entered the access_token."

# Create a new Chatbot instance with the following parameters:
# - conversation_id: None (can be a unique identifier for the conversation if desired)
# - parent_id: None (can be used to connect conversations if needed)
# - config: A dictionary containing configuration options, in this case, the access_token is provided
chatbot = Chatbot(conversation_id=None, 
                  parent_id=None,
                  config={
                      "access_token": access_token
                  })

In [5]:
def ask_chatGPT(p):
    """
    Generate and yield complete sentences from the output of the chatbot.

    Parameters:
        p (str): The prompt or input to be given to the chatbot.

    Yields:
        str: Complete sentences obtained from the chatbot's responses.
    """

    prev_text = ""    # Store the previously processed text.
    output_text = ""  # Accumulate the output from the chatbot.

    for data in chatbot.ask(p):
        message = data["message"][len(prev_text):]  # Extract new text from the response.
        output_text += message
        prev_text = data["message"]

        # Characters indicating the end of a sentence.
        complete_sentence_chars = ["。", "!", "?", "."]
        for char in complete_sentence_chars:
            if char in output_text:
                sentences = output_text.split(char)
                for sentence in sentences[:-1]:
                    yield sentence.strip() + char  # Yield complete sentences.
                output_text = sentences[-1]
                break

    # Yield the last incomplete sentence, if any.
    if output_text:
        yield output_text.strip()

## 3. Helper Functions

In [6]:
# Global speech recognition instance
asr = sr.Recognizer()

def speech_to_text(audio: sr.AudioData) -> tuple[bool, Union[str, Type[Exception]]]:
    """
    Convert speech audio to text using Google Web Speech API.
    
    Parameters:
        audio (sr.AudioData): Speech audio data.
        
    Returns:
        Tuple[bool, Union[str, Type[Exception]]]: A tuple containing a boolean indicating if the recognition
                                                 was successful (True) or not (False), and the recognized text
                                                 or the class of the exception if an error occurred.
    """
    global asr
    try:
        # Use Google Web Speech API to recognize speech from audio data
        return True, asr.recognize_google(audio, language="en-US")
    except Exception as e:
        # If an error occurs during speech recognition, return False and the type of the exception
        return False, e.__class__


def get_tts_data(text: str) -> bytes:
    """
    Generate Text-to-Speech (TTS) audio in mp3 format.
    
    Parameters:
        text (str): The text to be converted to speech.
        
    Returns:
        bytes: TTS audio in mp3 format.
    """
    # Create a BytesIO object to hold the TTS audio data in mp3 format
    tts_result = io.BytesIO()
    # Generate TTS audio using gTTS library with the specified text and language (en-US)
    tts = gTTS(text=text, lang='en-US', slow=False, tld='com.au')
    # Write the TTS audio data to the BytesIO object
    tts.write_to_fp(tts_result)
    tts_result.seek(0)
    # Read and return the TTS audio data as bytes
    return tts_result.read()

def tts_to_wav(tts_byte: bytes, framerate: int = 22050) -> np.ndarray:
    """
    Convert TTS audio from mp3 format to WAV format and set the desired frame rate and channels.
    
    Parameters:
        tts_byte (bytes): TTS audio in mp3 format.
        framerate (int, optional): Desired frame rate for the WAV audio. Defaults to 22050.
        
    Returns:
        numpy.ndarray: TTS audio in WAV format as a numpy array of float32 values.
    """
    # Convert the TTS audio bytes in mp3 format to a pydub AudioSegment object
    seg = AudioSegment.from_mp3(io.BytesIO(tts_byte))
    # Set the frame rate and number of channels for the audio
    seg = seg.set_frame_rate(framerate)
    seg = seg.set_channels(1)
    # Create a BytesIO object to hold the WAV audio data
    wavIO = io.BytesIO()
    # Export the AudioSegment as WAV audio to the BytesIO object
    seg.export(wavIO, format="wav")
    # Read the WAV audio data from the BytesIO object using scipy.io.wavfile.read()
    rate, wav = read(io.BytesIO(wavIO.getvalue()))
    return wav

def wav_to_numpy_float32(wav_byte: bytes) -> np.ndarray:
    """
    Convert WAV audio from bytes to a numpy array of float32 values.
    
    Parameters:
        wav_byte (bytes): WAV audio data.
        
    Returns:
        numpy.ndarray: WAV audio as a numpy array of float32 values.
    """
    # Convert the WAV audio bytes to a numpy array of float32 values
    return wav_byte.astype(np.float32, order='C') / 32768.0

def get_tts_numpy_audio(text: str) -> np.ndarray:
    """
    Generate Text-to-Speech (TTS) audio in WAV format and convert it to a numpy array of float32 values.
    
    Parameters:
        text (str): The text to be converted to speech.
        
    Returns:
        numpy.ndarray: TTS audio as a numpy array of float32 values.
    """
    # Generate TTS audio in mp3 format from the given text
    mp3_byte = get_tts_data(text)
    # Convert the TTS audio in mp3 format to WAV format and a numpy array of float32 values
    wav_byte = tts_to_wav(mp3_byte)
    return wav_to_numpy_float32(wav_byte)

def make_avatar_speaks(text: str) -> None:
    """
    Make the avatar speak the given text by pushing the audio track to the NVIDIA A2F instance.
    
    Parameters:
        text (str): The text to be spoken by the avatar.
        
    Returns:
        None
    """
    global a2f_url
    global sample_rate
    global a2f_avatar_instance
    # Get the TTS audio in WAV format as a numpy array of float32 values
    tts_audio = get_tts_numpy_audio(text)
    # Push the TTS audio to the NVIDIA A2F instance for the avatar to speak
    push_audio_track(a2f_url, tts_audio, sample_rate, a2f_avatar_instance)
    return      
        
    

In [7]:
import os
# using ibm instead of google

import numpy as np
import io
import wave

def wav_to_numpy_float32_new(wav_byte: bytes) -> np.ndarray:
    """
    Convert WAV audio bytes to a numpy array of float32 values.

    Parameters:
        wav_byte (bytes): The WAV audio in bytes.

    Returns:
        numpy.ndarray: The WAV audio as a numpy array of float32 values.
    """
    # Use a BytesIO stream to read the WAV bytes
    with io.BytesIO(wav_byte) as wav_file:
        # Open the WAV file
        with wave.open(wav_file, 'rb') as wav:
            # Read audio frames and convert to numpy array
            frame_count = wav.getnframes()
            audio_data = wav.readframes(frame_count)
            audio_array = np.frombuffer(audio_data, dtype=np.int16)

            # Normalize to float32
            return audio_array.astype(np.float32) / 32768.0


from ibm_watson import TextToSpeechV1
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator
apikey = os.getenv("IBM_WATSON_TTS_APIKEY")
url = os.getenv("IBM_WATSON_TTS_URL")
assert apikey is not None, "Set IBM_WATSON_TTS_APIKEY before running this cell."
assert url is not None, "Set IBM_WATSON_TTS_URL before running this cell."
# Setup Service
authenticator = IAMAuthenticator(apikey)
tts = TextToSpeechV1(authenticator=authenticator)
tts.set_service_url(url)
   
    
def ibm_tts(text):
    res = tts.synthesize(text, accept='audio/wav', voice='en-US_AllisonExpressive').get_result()
    audio_content = res.content
    
    push_audio_track(a2f_url, wav_to_numpy_float32_new(audio_content), sample_rate, a2f_avatar_instance)
    
    return 

In [8]:
import os
from ibm_watson import TextToSpeechV1
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator
apikey = os.getenv("IBM_WATSON_TTS_APIKEY")
url = os.getenv("IBM_WATSON_TTS_URL")
assert apikey is not None, "Set IBM_WATSON_TTS_APIKEY before running this cell."
assert url is not None, "Set IBM_WATSON_TTS_URL before running this cell."
# Setup Service
authenticator = IAMAuthenticator(apikey)
tts = TextToSpeechV1(authenticator=authenticator)
tts.set_service_url(url)
   
    
def ibm_tts(text):
    res = tts.synthesize(text, accept='audio/wav', voice='en-US_AllisonExpressive').get_result()
    audio_content = res.content
    
    push_audio_track(a2f_url, wav_to_numpy_float32(audio_content), sample_rate, a2f_avatar_instance)
    
    return 

## Avatar demo
***Make sure Omniverse audio2face is running in streaming mode to interact with our program.***

### Talk to your avatar

In [9]:
# Define the default audio2face URL and port.
default_url = 'localhost'
default_port = 50051

# Set the audio frame rate for the audio data.
sample_rate = 22050  # Replace '22050' with the desired audio frame rate (samples per second).

# Specify the instance name of the avatar in audio2face service.
a2f_avatar_instance = '/World/audio2face/PlayerStreaming'

# Define a variable for the port (you can change the value as needed).
# For example, if you want to use a different port, modify the value of 'port' accordingly.
port = default_port

# Create the complete audio2face URL by combining the URL and port variables.
a2f_url = f'{default_url}:{port}'

In [ ]:
# Set OPENAI_API_KEY in your environment before running this cell.
import os
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")
assert openai.api_key is not None, "Set OPENAI_API_KEY before running this cell."

def gpt3(stext):
    response = openai.Completion.create(
    engine = "text-davinci-003",
    prompt= stext,
    temperature=0.7,
    max_tokens=15,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0
    )
    content = response.choices[0].text.split('.')
    return response.choices[0].text



In [ ]:
# Set OPENAI_API_KEY in your environment before running this cell.
import os
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")
assert openai.api_key is not None, "Set OPENAI_API_KEY before running this cell."


#prompt:  "Imitate Hilary,  who is a 8-year-old girl whose intellectual, emotional, social, and physical development appears normal for her age. Hilary lives with her biological parents and a younger brother. Hilary has several close friends at school, one of whom is called Janet. Janet lives with her biological parents and her grandfather who they call Pop. A few days ago, Hilary told her mother that Pop did rude things to her and touched her girlie bits when they were in the pool out in the garden of Janet\u2019s house. she is here to talk to about that incident today.#Interviewer: Hi Hilary, How are you today?#Child: I am fine.#Interviewer: My job is to talk to children about things that happened to them.#Child: Okay.#",


def gpt3_Hilary (stext):
    response = openai.Completion.create(
    engine = "davinci:ft-simula-2023-01-16-16-34-07",
    prompt= stext,
    max_tokens=15,
    n=1,
    temperature=0.7,
    stop=["#", "Interviewer"]
    )
    
    return response.choices[0].text
        
    
    

In [24]:
with ignoreStderr():
    with sr.Microphone() as source:
        asr.adjust_for_ambient_noise(source, duration=5)
        while True:
            print('Say something')
            audio=asr.listen(source)
            is_valid_input, pp = speech_to_text(audio)
            if is_valid_input:
                print("User : ", pp)
                sentence = gpt3(pp)
                print("Avatar : ", sentence)
                #
                make_avatar_speaks(sentence)


Say something
Say something
User :  how are you
Avatar :  ?

I'm doing well, thanks for asking. How about you
Sending audio data...
SUCCESS
Closed channel
Say something
User :  what is your favourite colour what is your favourite colour
Avatar :  

My favorite color is blue.
Sending audio data...
SUCCESS
Closed channel
Say something


KeyboardInterrupt: 